In [10]:
# Cell 1: Install
import subprocess
subprocess.run(['pip', 'install', 'segmentation-models-pytorch', 'albumentations',
                'rasterio', 'timm', '-q'])
print('Install done')

Install done


In [11]:
# Cell 2: Seed
import random, numpy as np, torch, os
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)
set_seed(SEED)
print(f'Seed: {SEED}')

Seed: 42


In [12]:
# Cell 3: Download data
import os, subprocess
if not os.path.exists('/kaggle/working/data/data'):
    subprocess.run(['kaggle','competitions','download',
                    '-c','anrfaisehack-theme-1-phase2',
                    '-p','/kaggle/working'], check=True)
    subprocess.run(['unzip','-q',
                    '/kaggle/working/anrfaisehack-theme-1-phase2.zip',
                    '-d','/kaggle/working/data'], check=True)
    print('Data ready.')
else:
    print('Data already exists.')

Data already exists.


In [13]:
# Cell 4: Paths
from pathlib import Path
BASE_DIR   = Path('/kaggle/working/data/data')
IMAGE_DIR  = BASE_DIR / 'image'
LABEL_DIR  = BASE_DIR / 'label'
SPLIT_DIR  = BASE_DIR / 'split'
PRED_DIR   = BASE_DIR / 'prediction' / 'image'
OUTPUT_DIR = Path('/kaggle/working')

def read_split(fname):
    with open(SPLIT_DIR / fname) as f:
        return [line.strip() for line in f if line.strip()]

train_ids = read_split('train.txt')
val_ids   = read_split('val.txt')
all_ids   = train_ids + val_ids
# Always read test ids directly from files
test_ids  = [p.stem.replace('_image','') for p in sorted(PRED_DIR.glob('*_image.tif'))]
print(f'Train:{len(train_ids)} Val:{len(val_ids)} Test:{len(test_ids)} All:{len(all_ids)}')

Train:59 Val:10 Test:19 All:69


In [14]:
# Cell 5: Utilities
import pandas as pd, torch.nn as nn, torch.nn.functional as F
import rasterio, warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

NUM_CLASSES    = 3
TOTAL_CHANNELS = 12

def load_tif(path):
    with rasterio.open(path) as src:
        data = src.read()
    return np.transpose(data.astype(np.float32), (1, 2, 0))

def load_label(path):
    with rasterio.open(path) as src:
        lbl = src.read(1)
    return np.clip(lbl.astype(np.int64), 0, 2)

def normalize_image(img):
    out = np.zeros_like(img, dtype=np.float32)
    for c in range(img.shape[2]):
        ch = img[:, :, c]
        valid = ch[np.isfinite(ch)]
        if len(valid) == 0: continue
        p2, p98 = np.percentile(valid, 2), np.percentile(valid, 98)
        if p98 > p2:
            out[:, :, c] = np.clip((ch - p2) / (p98 - p2 + 1e-8), 0, 1)
    return out

def add_derived_channels(img):
    sar_hh = img[:,:,0]; sar_hv = img[:,:,1]
    green  = img[:,:,2]; red    = img[:,:,3]
    nir    = img[:,:,4]; swir   = img[:,:,5]
    ndwi      = (green - nir)  / (green + nir  + 1e-8)
    mndwi     = (green - swir) / (green + swir + 1e-8)
    ndvi      = (nir - red)    / (nir + red    + 1e-8)
    sar_ratio = sar_hh / (sar_hv + 1e-8)
    sar_diff  = sar_hh - sar_hv
    sar_sum   = sar_hh + sar_hv
    def norm01(x):
        mn, mx = x.min(), x.max()
        return np.clip((x-mn)/(mx-mn+1e-8),0,1) if mx>mn else np.zeros_like(x)
    extras = np.stack([
        norm01((ndwi+1)/2), norm01((mndwi+1)/2),
        norm01((ndvi+1)/2), norm01(sar_ratio),
        norm01(sar_diff),   norm01(sar_sum),
    ], axis=2)
    return np.concatenate([img, extras], axis=2)

def compute_miou(pred_mask, true_mask, num_classes=3):
    ious = []
    for cls in range(num_classes):
        pb = (pred_mask==cls).astype(np.uint8)
        tb = (true_mask==cls).astype(np.uint8)
        inter = (pb & tb).sum()
        union = (pb | tb).sum()
        ious.append(1.0 if union==0 else inter/union)
    return np.mean(ious), ious

print('Utilities ready')

Device: cuda
Utilities ready


In [15]:
# Cell 6: Dataset + Transforms
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

class FloodDataset(Dataset):
    def __init__(self, ids, image_dir, label_dir=None, transform=None, is_test=False):
        self.ids       = ids
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir) if label_dir else None
        self.transform = transform
        self.is_test   = is_test
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = load_tif(self.image_dir / f'{img_id}_image.tif')
        img = normalize_image(img)
        img = add_derived_channels(img)
        if self.is_test:
            if self.transform: img = self.transform(image=img)['image']
            return img, img_id
        lbl = load_label(self.label_dir / f'{img_id}_label.tif')
        if self.transform:
            aug = self.transform(image=img, mask=lbl)
            img, lbl = aug['image'], aug['mask']
        return img, lbl.long()

train_transform = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(translate_percent=0.15, scale=(0.85,1.25), rotate=(-45,45), p=0.6),
    A.ElasticTransform(alpha=120, sigma=6, p=0.4),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
    A.GaussNoise(p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.CoarseDropout(num_holes_range=(8,12), hole_height_range=(32,48),
                    hole_width_range=(32,48), p=0.4),
    A.RandomGamma(gamma_limit=(70,130), p=0.3),
    A.PadIfNeeded(min_height=512, min_width=512, border_mode=0),
    ToTensorV2(),
])
val_transform = A.Compose([
    A.PadIfNeeded(min_height=512, min_width=512, border_mode=0),
    ToTensorV2(),
])
print('Dataset ready')

Dataset ready


In [16]:
# Cell 7: Loss
import segmentation_models_pytorch as smp

class FloodFocusedLoss(nn.Module):
    def __init__(self, num_classes=3, dice_weight=0.7, gamma=2.0):
        super().__init__()
        self.num_classes   = num_classes
        self.dice_weight   = dice_weight
        self.gamma         = gamma
        self.class_weights = torch.tensor([1.0, 8.0, 2.0]).to(DEVICE)
        self.dice_cls_w    = [0.1, 0.8, 0.1]

    def focal_loss(self, pred, target):
        ce = F.cross_entropy(pred, target,
                             weight=self.class_weights, reduction='none')
        pt = torch.exp(-ce)
        return ((1-pt)**self.gamma * ce).mean()

    def dice_loss(self, pred, target):
        ps = F.softmax(pred, dim=1)
        total = 0
        for cls in range(self.num_classes):
            p = ps[:,cls].contiguous().view(-1)
            t = (target==cls).float().contiguous().view(-1)
            inter = (p*t).sum()
            total += self.dice_cls_w[cls]*(1-(2*inter+1)/(p.sum()+t.sum()+1))
        return total

    def forward(self, pred, target):
        return (1-self.dice_weight)*self.focal_loss(pred,target) + \
               self.dice_weight*self.dice_loss(pred,target)

print('Loss ready')

Loss ready


In [17]:
# Cell 8: Cross-validation + 4-model training
from torch.cuda.amp import GradScaler, autocast
from sklearn.model_selection import KFold

# 4 diverse architectures
MODEL_CONFIGS = [
    {'name': 'UNet++B5',  'arch': 'UnetPlusPlus', 'encoder': 'efficientnet-b5', 'seed': SEED},
    {'name': 'UNet++B4',  'arch': 'UnetPlusPlus', 'encoder': 'efficientnet-b4', 'seed': SEED+1},
    {'name': 'DeepLabB4', 'arch': 'DeepLabV3Plus','encoder': 'efficientnet-b4', 'seed': SEED+2},
    {'name': 'UnetMiTB2', 'arch': 'Unet',         'encoder': 'mit_b2',          'seed': SEED+3},  # ← Unet not UnetPlusPlus
]

def build_model(cfg):
    set_seed(cfg['seed'])
    kwargs = dict(
        encoder_name=cfg['encoder'],
        encoder_weights='imagenet',
        in_channels=TOTAL_CHANNELS,
        classes=NUM_CLASSES,
        activation=None,
    )
    if cfg['arch'] == 'UnetPlusPlus':
        kwargs['decoder_attention_type'] = 'scse'
        return smp.UnetPlusPlus(**kwargs).to(DEVICE)
    elif cfg['arch'] == 'Unet':
        kwargs['decoder_attention_type'] = 'scse'
        return smp.Unet(**kwargs).to(DEVICE)          # ← standard Unet for MiT
    else:
        return smp.DeepLabV3Plus(**kwargs).to(DEVICE)

def train_one_fold(model, tr_ids, va_ids, model_name, save_path,
                   epochs=100, lr=2e-4, patience=25):
    train_ds = FloodDataset(tr_ids, IMAGE_DIR, LABEL_DIR, transform=train_transform)
    val_ds   = FloodDataset(va_ids, IMAGE_DIR, LABEL_DIR, transform=val_transform)
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False,
                              num_workers=2, pin_memory=True)

    criterion = FloodFocusedLoss().to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=10)  # ← fixed
    scaler = GradScaler()

    best_flood = 0; no_improve = 0

    for epoch in range(1, epochs+1):
        model.train(); train_loss = 0
        for imgs, masks in train_loader:
            imgs, masks = imgs.float().to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(imgs), masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            train_loss += loss.item()

        model.eval()
        val_flood = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.float().to(DEVICE), masks.to(DEVICE)
                with autocast(): preds = model(imgs)
                pred_cls = torch.argmax(preds, dim=1).cpu().numpy()
                gt = masks.cpu().numpy()
                for p, g in zip(pred_cls, gt):
                    _, ious = compute_miou(p, g)
                    val_flood += ious[1]

        avg_flood = val_flood / len(val_ds)
        scheduler.step(avg_flood)

        if epoch % 20 == 0 or avg_flood > best_flood:
            print(f'  [{model_name}] Ep{epoch:03d} Loss:{train_loss/len(train_loader):.4f} FloodIoU:{avg_flood:.4f}')

        if avg_flood > best_flood:
            best_flood = avg_flood
            no_improve = 0
            torch.save(model.state_dict(), save_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  [{model_name}] Early stop ep{epoch} best_flood:{best_flood:.4f}')
                break

    model.load_state_dict(torch.load(save_path))
    return model, best_flood

# 5-Fold Cross Validation on all_ids
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_ids_arr = np.array(all_ids)

# Store all trained models and their weights
all_trained_models  = []
all_model_weights   = []

print(f'=== 5-FOLD CV x 4 MODELS ===')
print(f'Total models to train: {N_FOLDS * len(MODEL_CONFIGS)} ({N_FOLDS} folds x {len(MODEL_CONFIGS)} architectures)')

for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(all_ids_arr)):
    tr_ids = all_ids_arr[tr_idx].tolist()
    va_ids = all_ids_arr[va_idx].tolist()
    print(f'\n=== FOLD {fold_idx+1}/{N_FOLDS} | Train:{len(tr_ids)} Val:{len(va_ids)} ===')

    for cfg in MODEL_CONFIGS:
        save_path = OUTPUT_DIR / f'model_{cfg["name"]}_fold{fold_idx+1}.pth'
        model = build_model(cfg)
        model, best_flood = train_one_fold(
            model, tr_ids, va_ids,
            f'{cfg["name"]}_fold{fold_idx+1}',
            save_path, epochs=100, patience=25
        )
        all_trained_models.append(model)
        all_model_weights.append(best_flood)
        print(f'  {cfg["name"]} fold{fold_idx+1}: FloodIoU={best_flood:.4f}')
        # Free GPU memory between models
        torch.cuda.empty_cache()

print(f'\nAll {len(all_trained_models)} models trained!')
print(f'Avg flood IoU across all models: {np.mean(all_model_weights):.4f}')

=== 5-FOLD CV x 4 MODELS ===
Total models to train: 20 (5 folds x 4 architectures)

=== FOLD 1/5 | Train:55 Val:14 ===
  [UNet++B5_fold1] Ep001 Loss:1.0793 FloodIoU:0.1427
  [UNet++B5_fold1] Ep002 Loss:0.9937 FloodIoU:0.1547
  [UNet++B5_fold1] Ep003 Loss:0.9575 FloodIoU:0.1629
  [UNet++B5_fold1] Ep004 Loss:0.9218 FloodIoU:0.1805
  [UNet++B5_fold1] Ep005 Loss:0.8924 FloodIoU:0.2035
  [UNet++B5_fold1] Ep006 Loss:0.8270 FloodIoU:0.2255
  [UNet++B5_fold1] Ep008 Loss:0.7964 FloodIoU:0.2359
  [UNet++B5_fold1] Ep010 Loss:0.6945 FloodIoU:0.2444
  [UNet++B5_fold1] Ep011 Loss:0.7208 FloodIoU:0.2613
  [UNet++B5_fold1] Ep019 Loss:0.6899 FloodIoU:0.2652
  [UNet++B5_fold1] Ep020 Loss:0.6548 FloodIoU:0.2744
  [UNet++B5_fold1] Ep022 Loss:0.6360 FloodIoU:0.2869
  [UNet++B5_fold1] Ep023 Loss:0.6325 FloodIoU:0.2975
  [UNet++B5_fold1] Ep039 Loss:0.6084 FloodIoU:0.2988
  [UNet++B5_fold1] Ep040 Loss:0.6015 FloodIoU:0.3017
  [UNet++B5_fold1] Ep047 Loss:0.5900 FloodIoU:0.3095
  [UNet++B5_fold1] Ep049 Loss:0.6

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/98.9M [00:00<?, ?B/s]

  [UnetMiTB2_fold1] Ep001 Loss:1.0733 FloodIoU:0.1244
  [UnetMiTB2_fold1] Ep002 Loss:1.0210 FloodIoU:0.1278
  [UnetMiTB2_fold1] Ep007 Loss:0.9273 FloodIoU:0.1379
  [UnetMiTB2_fold1] Ep008 Loss:0.9138 FloodIoU:0.1488
  [UnetMiTB2_fold1] Ep009 Loss:0.8719 FloodIoU:0.1519
  [UnetMiTB2_fold1] Ep010 Loss:0.8679 FloodIoU:0.1555
  [UnetMiTB2_fold1] Ep013 Loss:0.8294 FloodIoU:0.1562
  [UnetMiTB2_fold1] Ep014 Loss:0.8025 FloodIoU:0.1572
  [UnetMiTB2_fold1] Ep016 Loss:0.7877 FloodIoU:0.1975
  [UnetMiTB2_fold1] Ep017 Loss:0.7710 FloodIoU:0.2106
  [UnetMiTB2_fold1] Ep018 Loss:0.7757 FloodIoU:0.2226
  [UnetMiTB2_fold1] Ep020 Loss:0.7696 FloodIoU:0.2108
  [UnetMiTB2_fold1] Ep021 Loss:0.7708 FloodIoU:0.2337
  [UnetMiTB2_fold1] Ep026 Loss:0.7504 FloodIoU:0.2398
  [UnetMiTB2_fold1] Ep027 Loss:0.7066 FloodIoU:0.2457
  [UnetMiTB2_fold1] Ep029 Loss:0.7167 FloodIoU:0.2530
  [UnetMiTB2_fold1] Ep030 Loss:0.6660 FloodIoU:0.2621
  [UnetMiTB2_fold1] Ep032 Loss:0.6776 FloodIoU:0.2654
  [UnetMiTB2_fold1] Ep038 Lo

KeyboardInterrupt: 

In [18]:
# Cell 9: Normalize weights and validate ensemble
print('=== ENSEMBLE VALIDATION ===')

# Softmax weights so they sum to 1
weights_arr = np.array(all_model_weights)
weights_arr = weights_arr / weights_arr.sum()
print(f'Model weights (normalized by flood IoU):')
for i, (w, m) in enumerate(zip(weights_arr, all_trained_models)):
    fold = i % N_FOLDS + 1
    arch = MODEL_CONFIGS[i // N_FOLDS]['name']
    print(f'  {arch}_fold{fold}: {w:.4f}')

for m in all_trained_models: m.eval()

# Validate on original val split
val_ds_e  = FloodDataset(val_ids, IMAGE_DIR, LABEL_DIR, transform=val_transform)
val_ldr_e = DataLoader(val_ds_e, batch_size=2, shuffle=False, num_workers=2)

total_miou = 0; class_ious = [0,0,0]
with torch.no_grad():
    for imgs, masks in val_ldr_e:
        imgs = imgs.float().to(DEVICE)
        ensemble = None
        with autocast():
            for m, w in zip(all_trained_models, weights_arr):
                p = F.softmax(m(imgs), dim=1).cpu().numpy()
                ensemble = p*w if ensemble is None else ensemble + p*w
        pred_cls = np.argmax(ensemble, axis=1)
        gt = masks.numpy()
        for p, g in zip(pred_cls, gt):
            miou, ious = compute_miou(p, g)
            total_miou += miou
            for i in range(3): class_ious[i] += ious[i]

n = len(val_ds_e)
print(f'\nEnsemble Validation Results:')
print(f'  No Flood IoU  : {class_ious[0]/n:.4f}')
print(f'  Flood IoU     : {class_ious[1]/n:.4f}')
print(f'  Water Body IoU: {class_ious[2]/n:.4f}')
print(f'  Mean IoU      : {total_miou/n:.4f}')

=== ENSEMBLE VALIDATION ===
Model weights (normalized by flood IoU):
  UNet++B5_fold1: 0.0781
  UNet++B5_fold2: 0.0767
  UNet++B5_fold3: 0.0617
  UNet++B5_fold4: 0.0738
  UNet++B5_fold5: 0.0620
  UNet++B4_fold1: 0.0612
  UNet++B4_fold2: 0.0532
  UNet++B4_fold3: 0.0551
  UNet++B4_fold4: 0.0879
  UNet++B4_fold5: 0.0815
  DeepLabB4_fold1: 0.0822
  DeepLabB4_fold2: 0.0871
  DeepLabB4_fold3: 0.0682
  DeepLabB4_fold4: 0.0713

Ensemble Validation Results:
  No Flood IoU  : 0.7421
  Flood IoU     : 0.1966
  Water Body IoU: 0.3793
  Mean IoU      : 0.4393


In [19]:
# Cell 10: TTA + Ensemble prediction
def predict_tta_ensemble(img_tensor, models, weights):
    """8-way TTA x all models ensemble"""
    img = img_tensor.unsqueeze(0).float().to(DEVICE)

    def run_ensemble(x):
        probs = None
        for m, w in zip(models, weights):
            with torch.no_grad():
                with autocast():
                    p = F.softmax(m(x), dim=1).squeeze().cpu().numpy()
            probs = p*w if probs is None else probs + p*w
        return probs

    augs = []
    augs.append(run_ensemble(img))
    augs.append(np.flip(run_ensemble(torch.flip(img,[3])), axis=2))
    augs.append(np.flip(run_ensemble(torch.flip(img,[2])), axis=1))
    augs.append(np.rot90(run_ensemble(torch.rot90(img,1,[2,3])), -1, axes=(1,2)))
    augs.append(np.rot90(run_ensemble(torch.rot90(img,2,[2,3])), -2, axes=(1,2)))
    augs.append(np.rot90(run_ensemble(torch.rot90(img,3,[2,3])), -3, axes=(1,2)))
    augs.append(np.flip(np.flip(run_ensemble(torch.flip(img,[2,3])),axis=1),axis=2))
    r = torch.flip(torch.rot90(img,1,[2,3]),[3])
    augs.append(np.rot90(np.flip(run_ensemble(r),axis=2),-1,axes=(1,2)))

    return np.argmax(np.mean(augs, axis=0), axis=0)


def mask_to_rle(mask):
    mask = mask.flatten(order='F')
    pixels = np.concatenate([[0], mask, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(r) for r in runs) if len(runs) else '0 0'


for m in all_trained_models: m.eval()

test_ds = FloodDataset(test_ids, PRED_DIR, label_dir=None,
                       transform=val_transform, is_test=True)
rows = []
print(f'Generating predictions: 8-way TTA x {len(all_trained_models)} models ({len(test_ids)} images)...')
for img_t, img_id in test_ds:
    class_map  = predict_tta_ensemble(img_t, all_trained_models, weights_arr)
    flood_mask = (class_map == 1).astype(np.uint8)
    flood_px   = flood_mask.sum()
    rle = mask_to_rle(flood_mask) if flood_px > 0 else '0 0'
    rows.append({'id': img_id, 'rle_mask': rle})
    print(f'  {img_id} | no_flood:{(class_map==0).sum()} flood:{flood_px} water:{(class_map==2).sum()}')

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print('submission.csv saved!')
print(df.head())

Generating predictions: 8-way TTA x 14 models (19 images)...
  20240529_EO4_RES2_fl_pid_080 | no_flood:240587 flood:347 water:21210
  20240529_EO4_RES2_fl_pid_081 | no_flood:214706 flood:4388 water:43050
  20240529_EO4_RES2_fl_pid_082 | no_flood:239131 flood:22042 water:971
  20240529_EO4_RES2_fl_pid_083 | no_flood:180044 flood:64717 water:17383
  20240529_EO4_RES2_fl_pid_084 | no_flood:68760 flood:150067 water:43317
  20240529_EO4_RES2_fl_pid_085 | no_flood:71801 flood:129787 water:60556
  20240529_EO4_RES2_fl_pid_086 | no_flood:173150 flood:44301 water:44693
  20240529_EO4_RES2_fl_pid_087 | no_flood:257609 flood:2053 water:2482
  20240529_EO4_RES2_fl_pid_088 | no_flood:236524 flood:625 water:24995
  20240529_EO4_RES2_fl_pid_089 | no_flood:210707 flood:40725 water:10712
  20240529_EO4_RES2_fl_pid_090 | no_flood:207038 flood:51392 water:3714
  20240529_EO4_RES2_fl_pid_091 | no_flood:255207 flood:142 water:6795
  20240529_EO4_RES2_fl_pid_092 | no_flood:197994 flood:6239 water:57911
  20